# Test Set Features + Predicciones
Construye la matriz de features para `train_v2.csv` (predicción marzo 2017)
y genera probabilidades de churn con el mejor modelo entrenado.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.features.build_test_features_09 import build_and_save_test
from src.models.train_06 import FEATURE_COLS, MODELS_DIR, evaluate_on_test

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Construir features del test set

In [ ]:
df_test = build_and_save_test()
df_test.head()

## 2. Cargar mejor modelo y generar predicciones

In [ ]:
artifact = joblib.load(MODELS_DIR / 'best_model.joblib')
model = artifact['model']
print(f'Modelo: {artifact["name"]}')

X_test = df_test[FEATURE_COLS].values.astype('float32')
y_test = df_test['is_churn'].values

proba = model.predict_proba(X_test)[:, 1]
print(f'Predicciones generadas: {len(proba):,}')
print(f'Score medio: {proba.mean():.4f}')

## 3. Evaluación en test set (train_v2)

In [ ]:
from src.models.train_06 import evaluate_on_test
result = evaluate_on_test(artifact['name'], model, X_test, y_test)

## 4. Distribución de scores de predicción

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(proba[y_test == 0], bins=60, alpha=0.6, color='#4CAF50', label='Renewal', density=True)
axes[0].hist(proba[y_test == 1], bins=60, alpha=0.6, color='#F44336', label='Churn', density=True)
axes[0].axvline(result['threshold'], color='black', linestyle='--',
                label=f'Umbral={result["threshold"]:.2f}')
axes[0].set_title('Distribución de scores — train_v2 (test set)')
axes[0].set_xlabel('P(churn)')
axes[0].legend()

df_pred = df_test[['msno', 'is_churn']].copy()
df_pred['proba_churn'] = proba
top_risk = df_pred.nlargest(20, 'proba_churn')
axes[1].barh(range(20), top_risk['proba_churn'].values[::-1], color='#F44336')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([f'...{m[-6:]}' for m in top_risk['msno'].values[::-1]], fontsize=7)
axes[1].set_title('Top 20 usuarios con mayor riesgo de churn')
axes[1].set_xlabel('P(churn)')
plt.tight_layout()
plt.show()

## 5. Exportar predicciones

In [ ]:
DATA_PROC = ROOT / 'data' / 'processed'
df_pred = df_test[['msno', 'is_churn']].copy()
df_pred['proba_churn'] = proba
df_pred['pred_churn'] = (proba >= result['threshold']).astype(int)

df_pred.to_parquet(DATA_PROC / 'predictions_test.parquet', index=False)
print(f'Predicciones guardadas: {len(df_pred):,} usuarios')
df_pred.head(10)